<a href="https://colab.research.google.com/github/siva123995/ProjectLLM/blob/main/Need_of_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -qU langgraph langchain_community langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.5 MB/s eta 0:00:00


In [21]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('Secret')
# GEMINI_API_KEY=userdata.get('sivathede')

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = "models/gemini-3-flash-preview", google_api_key=GEMINI_API_KEY)

In [23]:
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, AIMessage # Import AIMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

def chat(state: AgentState):
  message = state['messages']
  response = model.invoke(message)
  return {"messages": [AIMessage(content=response.content)]} # Wrap response in AIMessage

graph = StateGraph(AgentState)

graph.add_node("chat",chat)
graph.add_edge(START, "chat")
graph.add_edge("chat",END)

agent = graph.compile()
response= agent.invoke({"messages":[HumanMessage("My name is Venkata")]})
print(response)

ChatGoogleGenerativeAIError: Error calling model 'models/gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 35.39408586s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM

In [14]:
response= agent.invoke({"messages":[HumanMessage("What is My name")]})
print(response)

{'messages': [HumanMessage(content='What is My name', additional_kwargs={}, response_metadata={}, id='d8309a68-b254-4412-919d-702bf29840a0'), AIMessage(content=[{'type': 'text', 'text': 'I don’t know your name unless you tell me! As an AI, I don’t have access to your personal identity or private information.\n\nWhat would you like me to call you?', 'extras': {'signature': 'EpsHCpgHAQw51se9AKTa94iA/OradCecC+8kLkzwSQTTdz+mnkAZOZ94jablPQpMwR7cyhkIFWffxLvTq3V2eAmV5GP470VExeX1DsNoQVU25493wyKeytDLp1fq05//EVtspVYmZatJR7AQtTcqFST8xAJ8RfNUZ55vgJoCtfvuiJYwjVRasDc+JyrKCWQ4AUxC6U5T+sYO4AEgRkCmJqM6pkyPwpN/aPK+U0A2Td/JgqwUoum94WEQmpProaTBGjDvgsrMyh9Fxva2rQHospekHminr10ndMJ56J1Eaik78nXOmjvoHuc+g7Bjh8rdMqpichalzVNpnmf1ZuqHTxbWTfatTCIKsW4jmc+pO1KBq3LEJbcmDLDs+j0wumw396piHLVoITUn+UXKc8p+qOXeFl+eay+eGVf60nDTPfh33zECTpmIeUXOjPFjFYQbOYcZxpD8PNnd2DhKfr9AYI+Pt62IsjQk/megTIxUAOoxDxWe2mc2hs0KmgSC6beDCCVTB0d1ExgCqHEEu9F+8IJ2fJblGxqQFEi/xM6pUginC2MzHF466zZivmUs9lUSabP7UU0/X1ebZ4p+5whWquDAaja0RzHUFvwyipgoYetoME+0

## LangChain Agent Interaction Examples

Here are two examples of interactions with the LangChain agent:

### Example 1: Introduction

**User Input:**
```
My name is Venkata
```

**Agent Response:**
```
Hello Venkata! It's nice to meet you. How can I help you today?
```

### Example 2: Name Recall

**User Input:**
```
What is My name
```

**Agent Response:**
```
I don’t know your name unless you tell me! As an AI, I don’t have access to your personal identity or private information. What would you like me to call you?
```

In [17]:
# Agent isn't storing the information, So, need to memory is necessary, let's implement memory

In [24]:
from IPython.core.alias import Configurable
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, AIMessage # Import AIMessage
from langgraph.checkpoint.memory import MemorySaver

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

def chat(state: AgentState):
  message = state['messages']
  response = model.invoke(message)
  return {"messages": [AIMessage(content=response.content)]} # Wrap response in AIMessage

graph = StateGraph(AgentState)
memory = MemorySaver()

graph.add_node("chat",chat)
graph.add_edge(START, "chat")
graph.add_edge("chat",END)

agent = graph.compile(checkpointer=memory)

configuration = {
    "configurable":{
        "thread_id":1
    }
}

response= agent.invoke({"messages":[HumanMessage("My name is Venkata")]}, config=configuration)
print(response)

ChatGoogleGenerativeAIError: Error calling model 'models/gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 52.541468799s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPer

In [ ]:
response= agent.invoke({"messages":[HumanMessage("What is my name?")]}, config=configuration)
print(response)